# 05 - Other TWPAs and segmented devices

The previous tutorials used a three-wave-mixing kinetic-inductance TWPA (KI-TWPA) with a uniform line. This
notebook builds other designs with the same tools:

1. **A four-wave-mixing (4WM) KI-TWPA with a resonant notch and periodic modulation**, from M. Malnou, *Synthesis of
   artificial transmission lines tailored for traveling-wave parametric processes*, Phys. Rev. Applied **25**,
   054016 (2026). A resonant phase-matching (rpm) filter in one cell per supercell creates a narrow notch whose
   dispersion compensates the Kerr phase mismatch of the pump (the technique of K. O'Brien *et al.*, PRL **113**,
   157001 (2014) and C. Macklin *et al.*, Science **350**, 307 (2015)), while a periodic modulation of the shunt
   capacitors opens a stopband that suppresses the pump third harmonic. The rpm cell is a user-defined model, which
   also shows how to make a custom cell work with the `bloch` model.
2. **Segmented devices**: real devices are rarely uniform. Film thickness and linewidth drift across the chip, and
   some designs vary the line on purpose. `twpasolver` describes such devices as a chain of uniform **segments**; we
   build one from the KI-TWPA holotype and check the segmented solver against the uniform one.
3. **Fabrication variations**: random steps of the line parameters from segment to segment.
4. **A Floquet-mode TWPA**: the nonlinearity is switched on and off smoothly along the line, so that the Floquet
   modes of the pumped line connect adiabatically to the linear modes at both ends and less signal is lost into
   frequency-conversion sidebands. The idea is from K. Peng *et al.*, *Floquet-Mode Traveling-Wave Parametric
   Amplifiers*, PRX Quantum **3**, 020306 (2022) (first experiment: arXiv:2503.11812, 2025).

Several runs here raise `passes` above its default. When doing so, raise `anderson_depth` with it (here
`passes - 10`), so that the Anderson acceleration can use the longer history.

In [ ]:
import time
from typing import Literal

import matplotlib.pyplot as plt
import numpy as np

from twpasolver import ModeSet, TWPAnalysis
from twpasolver.cmes.bloch import PORT_OUT
from twpasolver.cmes.solver_config import BlochSolverConfig
from twpasolver.logger import log
from twpasolver.matrices_arrays import ABCDArray
from twpasolver.models import TWPA, LCLfBaseCell
from twpasolver.twoport import TwoPortModel

log.setLevel("WARNING")
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

## 1. A 4WM KI-TWPA with a resonant notch and a modulated line

### The supercell

The line is a 50 $\Omega$ LC ladder of kinetic inductances $L_0 = 100$ pH, whose nonlinearity
$L = L_0\,[1 + (I/I_*)^2]$ with $I_* = 100\,\mu$A is twpasolver's TWPA model with no dc bias (pure 4WM). A supercell
holds 9 unit cells (Appendix I of the paper):

* eight cells with shunt capacitors $C_0 \dots C_4, C_4, C_3, C_2$ = 51.3, 48.7, 42.0, 34.3, 29.3, 29.3, 34.3,
  42.0 fF: two overlapped cosine modulations of the line impedance that open a stopband around 27 GHz;
* one rpm cell (shunt capacitor 48.7 fF) whose series inductor is shunted by a series LC branch
  ($L_r = 4.92$ nH, $C_r = 53.5$ fF).

The first eight are `LCLfBaseCell`s without finger inductance. The rpm cell is a user model: a `name`, a
`single_abcd` method and `bloch_nl_elements()`, which tells the `bloch` model that the nonlinear inductor $L_0$
carries the cell's output current. That is accurate away from the notch, where the LC branch is a high impedance;
near the pump it carries about 14 % more, in one cell out of nine, which we neglect.

In [ ]:
class RPMCell(TwoPortModel):
    """Shunt C, then a series inductor L in parallel with a series LC branch (rpm filter)."""

    name: Literal["RPMCell"] = "RPMCell"
    L: float
    C: float
    Lr: float
    Cr: float

    def single_abcd(self, freqs):
        w = 2 * np.pi * np.asarray(freqs, dtype=float)
        Z_branch = 1j * w * self.Lr + 1 / (1j * w * self.Cr)
        Zs = 1 / (1 / (1j * w * self.L) + 1 / Z_branch)
        Y = 1j * w * self.C
        abcd = np.empty((len(w), 2, 2), dtype=complex)
        abcd[:, 0, 0] = 1
        abcd[:, 0, 1] = Zs
        abcd[:, 1, 0] = Y
        abcd[:, 1, 1] = 1 + Y * Zs
        return ABCDArray(abcd)

    def bloch_nl_elements(self):
        return [(PORT_OUT, self.L)]


L0, Istar = 100e-12, 100e-6
caps = np.array([51.3, 48.7, 42.0, 34.3, 29.3, 29.3, 34.3, 42.0]) * 1e-15
C_rpm, Lr, Cr = 48.7e-15, 4.92e-9, 53.5e-15


def ktwpa(rpm: bool = True, modulated: bool = True, Ip0: float = 22e-6) -> TWPA:
    shunt = caps if modulated else np.full(8, np.mean([*caps, C_rpm]))
    last_C = C_rpm if modulated else shunt[0]
    cells = [LCLfBaseCell(L=L0, C=c, Lf=0.0) for c in shunt]
    cells.append(
        RPMCell(L=L0, C=last_C, Lr=Lr, Cr=Cr)
        if rpm
        else LCLfBaseCell(L=L0, C=last_C, Lf=0.0)
    )
    return TWPA(cells=cells, N=556, Istar=Istar, Idc=0.0, Ip0=Ip0)


an = TWPAnalysis(twpa=ktwpa(), freqs=(0.5, 45, 2e-3))
lin = an.linear
print(
    f"{an.twpa.N_tot} cells, Bloch impedance at 1 GHz: {lin.Zb[0, np.searchsorted(lin.freqs, 1.0)]:.1f} Ohm"
)
print("stopbands (GHz):", np.round(lin.stopbands, 2))
print(
    f"notch from (L0 + Lr) Cr: {1 / (2 * np.pi * np.sqrt((L0 + Lr) * Cr)) / 1e9:.2f} GHz"
)

### Linear response

The notch sits where the LC branch, capacitive below its own resonance, cancels the series inductor: an open in the
series branch at $1/(2\pi\sqrt{(L_0+L_r)C_r}) = 9.71$ GHz (the paper quotes 10 GHz). Just below it $k^*$ rises
steeply, which is the extra pump wavenumber the 4WM phase matching needs. The modulation stopband around 27 GHz
contains the pump third harmonic $3 f_p \approx 27.3$ GHz.

In [ ]:
fp, Ip0 = 9.1, 22e-6
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(6, 6), sharex=True)
ax1.plot(lin.freqs, lin.S21_db)
ax2.plot(lin.freqs, lin.k_star)
for ax in (ax1, ax2):
    for lo, hi in lin.stopbands:
        ax.axvspan(lo, hi, color="gray", alpha=0.3)
    ax.axvline(fp, color="C3", ls=":")
    ax.axvline(3 * fp, color="C3", ls="--")
ax1.set_ylim(-20, 2)
ax1.set_ylabel("$|S_{21}|$ (dB)")
ax2.set_ylabel("$k^*$ (rad/cell)")
ax2.set_xlabel("Frequency (GHz)")
plt.show()

### Phase matching and gain

`phase_matching` with `mixing="4WM"` includes the self- and cross-phase modulation of the pump. Its optimal pump
lies below the notch (8.9 GHz here, from the linear mismatch averaged over 4-8 GHz); we use the pump of the paper,
$f_p = 9.1$ GHz with $I_p = 22\,\mu$A.

Pump, signal and idler alone do not show what happens in this line: the pump third harmonic $3 f_p$ and the
up-conversion products $2 f_p + f_s$ and $2 f_p + f_i$ fall in or next to the modulation stopband, and they exchange
power with the pump, the signal and the idler. We therefore solve with an extended 4WM mode set from the start, for
signals on both sides of the pump.

In [ ]:
pm = an.phase_matching(
    (4, 8, 0.1), np.arange(8.6, 9.6, 0.02), mixing="4WM", modes="basic_4wm"
)
print(f"optimal pump from the linear phase matching: {pm.optimal_pump_freq:.2f} GHz")
pm.plot()
plt.show()

EXT4 = ModeSet(
    ["p", "s", "i", "p3", "p2s", "p2i"],
    [["i", "p+p-s"], ["p3", "p+p+p"], ["p2s", "p+p+s"], ["p2i", "p+p+i"]],
)
print(EXT4)
signals = np.linspace(4, 14, 205)
cfg_rpm = BlochSolverConfig(passes=60, anderson_depth=50)
fig, ax = plt.subplots(figsize=(7, 3.5))
for model in ("simple", "bloch"):
    r = an.gain(
        signals, pump=fp, Is0=1e-8, model=model, modes=EXT4, bloch_config=cfg_rpm
    )
    ax.plot(r.signal_freqs, r.gain_db, label=model)
    print(f"{model}: converged {r.converged.sum()}/{len(signals)}")
no_rpm = TWPAnalysis(twpa=ktwpa(rpm=False), freqs=(0.5, 45, 2e-3))
r = no_rpm.gain(
    signals, pump=fp, Is0=1e-8, model="bloch", modes=EXT4, bloch_config=cfg_rpm
)
ax.plot(r.signal_freqs, r.gain_db, label="bloch, without the rpm cell")
ax.set_xlabel("Signal frequency (GHz)")
ax.set_ylabel("Gain (dB)")
ax.legend()
plt.show()

With the `bloch` model the gain is about 17-19.5 dB over 4-7.5 GHz and, mirrored around the pump, over
10.5-14 GHz, close to the 20 dB from 4 to 8 GHz of the paper's harmonic-balance simulation. It falls where the signal
or the idler approaches the notch, between 8 and 10 GHz, apart from the degenerate point $f_s = f_p$. One signal near
7.4 GHz does not converge: there $2 f_p + f_s$ sits at the lower edge of the modulation stopband (25.8 GHz). Such
points are flagged in `GainResult.converged` and should not be trusted.

The `simple` model has no stopbands: it lets the third harmonic grow as a free traveling wave, which drains the
pump, and gives about 4 dB less gain. With pump, signal and idler only, the two models agree (17.8 and 18.2 dB on
average over 4-8 GHz), so the difference comes from the modes in and next to the stopband.

Without the rpm cell the Kerr mismatch is left uncompensated and the gain collapses to a single lobe of about 11 dB
around the pump: the notch is what makes the device an amplifier.

### Why the modulation: the pump third harmonic

In a KI-TWPA the line is nearly dispersionless far below its cutoff, so the pump third harmonic $3 f_p$ is almost
phase matched and grows along the line. We compare the design with a line whose shunt capacitors all take their
mean value (no modulation stopband). Without the stopband the up-conversion products $2 f_p \pm f_s$ are nearly
phase matched too and the power cascades to ever higher modes: no truncated mode set follows it, and with the
products of the extended set the forward-backward iteration does not converge. To isolate the harmonic, this
comparison uses pump, signal, idler and $3 f_p$ only. The harmonic lies inside the stopband of the design, a regime
that only the `bloch` model describes.

In [ ]:
P3 = ModeSet(["p", "s", "i", "p3"], [["i", "p+p-s"], ["p3", "p+p+p"]])
print(P3)
runs = {}
for name, twpa in (
    ("modulated (design)", ktwpa()),
    ("no modulation", ktwpa(modulated=False)),
):
    a = TWPAnalysis(twpa=twpa, freqs=(0.5, 45, 2e-3))
    r = a.gain(
        signals,
        pump=fp,
        Is0=1e-8,
        model="bloch",
        modes=P3,
        bloch_config=cfg_rpm,
    )
    third = 20 * np.log10(
        np.abs(r.b_out[:, r.mode("p3")]) / np.abs(r.b_out[:, r.mode("p")])
    )
    runs[name] = r
    print(
        f"{name}: mean gain {np.mean(r.gain_db):.1f} dB, third harmonic / pump at the output "
        f"{np.mean(third):.1f} dB, converged {r.converged.sum()}/{len(signals)}"
    )
for name, r in runs.items():
    r.plot_currents(title=name)
plt.show()

The stopband suppresses the third harmonic at the output by more than 20 dB (-30 against -6.5 dB relative to the
pump), in line with the 20-27 dB the paper reports, and the mode currents show the unsuppressed harmonic taking a
large share of the pump power along the line. In this four-mode model this also lowers the gain; the paper, with many
more mixing products, finds similar pump depletion with and without the stopband and attributes its role to keeping
the parametric process a clean 4WM one (a symmetric gain profile and a near-ideal quantum efficiency). Without the
modulation the iteration does not converge for about one signal in ten (residuals up to 0.13), a sign of the strong
harmonic generation; those points are flagged in `GainResult.converged`.

## 2. Segmented devices

`twpasolver` describes such devices as a chain of
uniform **segments**. We build a segmented device, check the segmented solver against the uniform one, and use it for random
fabrication variations and a Floquet-mode design.

We return to the KI-TWPA holotype `imskit_v3_holotype.json` (3WM, $I_{dc}$ bias, pump at 15.8 GHz,
just above the first stopband), with the extended 3WM mode set of notebook 04.

In [ ]:
FREQS = (0.05, 50, 2e-3)  # GHz; reaches the 3rd pump harmonic (47.4 GHz)
FP = 15.8  # pump frequency (GHz)
SIG = np.linspace(3, 12.5, 100)  # signal frequencies (GHz)
MODES = ModeSet.extended_3wm(
    n_pump_harmonics=2, n_frequency_conversion=2, n_signal_harmonics=0
)
CFG = BlochSolverConfig(passes=60, anderson_depth=50)

base = TWPA.from_file("imskit_v3_holotype.json")
print(
    f"{base.N} supercells of {base.N_tot // base.N:.0f} unit cells, {base.N_tot:.0f} unit cells in total"
)

### Building a segmented device

A segmented device is a `TWPA` whose `cells` are themselves `TWPA` objects. Each segment is a uniform
TWPA with its own unit cells, number of supercells `N` and nonlinearity (`Istar`, `Idc`); the outer
`TWPA` (with `N=1`) only chains them and carries the pump current and the reference impedance.
Each segment must hold a **whole number of supercells**: the Bloch modes of a segment are defined on
its supercell, so a segment cannot end in the middle of one. To cut the holotype into `n_seg` equal
segments, `n_seg` must divide its 1200 supercells.

`make_segmented` below copies the holotype's unit cells into `n_seg` segments and scales the line
inductance $L$ and the finger inductance $L_f$ of every unit cell of segment $j$ by `L_scale[j]` and
`Lf_scale[j]`.

In [ ]:
def make_segmented(n_seg, L_scale=None, Lf_scale=None):
    if base.N % n_seg:
        raise ValueError(
            f"n_seg={n_seg} must divide N={base.N} (whole supercells per segment)"
        )
    L_scale = np.ones(n_seg) if L_scale is None else L_scale
    Lf_scale = np.ones(n_seg) if Lf_scale is None else Lf_scale
    segments = []
    for j in range(n_seg):
        cells = [c.model_copy(deep=True) for c in base.cells]
        for c in cells:
            c.L *= L_scale[j]
            c.Lf *= Lf_scale[j]
        segments.append(base.model_copy(update={"cells": cells, "N": base.N // n_seg}))
    return TWPA(cells=segments, N=1, Ip0=base.Ip0, Z0_ref=base.Z0_ref)


def analysis(twpa):
    return TWPAnalysis(twpa=twpa, freqs=FREQS)


an = analysis(make_segmented(4))
lin = an.linear
print("k shape (n_segments, n_freq):", lin.k.shape)
print("segment bounds (unit cells):", lin.seg_bounds)
print("segment midpoints (unit cells):", lin.seg_midpoints)

`an.linear` stores the Bloch wavenumber `k`, attenuation `alpha`, Bloch impedance `Zb` and its
`reflection_coeff` **per segment**, with shape `(n_segments, n_freq)` (a uniform device is one
segment). `S21`, `S11` and `abcd` are those of the whole chain, computed by multiplying the
segment ABCD matrices, so they already contain the reflections at the junctions. `stopbands` lists the
stopbands of the *first* segment only.

The two gain models treat the junctions differently:

* `model="bloch"` solves each segment on its own forward and backward Bloch modes and joins consecutive
  segments with **exact junction conditions**, from continuity of voltage and current at the shared
  supercell boundary. A step in the line parameters therefore reflects part of every wave, and these
  reflections interfere with each other and with the ports.
* `model="simple"` has forward waves only. It interpolates the per-segment wavenumbers (and the
  nonlinearity) **linearly between segment midpoints**, so it sees a smooth taper without reflections.
  It describes how phase matching drifts along the line, not the reflections.

### Check: identical segments and runtime

Cutting the holotype into identical segments changes nothing physically, so the `bloch` gain must
equal that of the uniform device: the junctions between identical segments are transparent. We
compare a few `n_seg` and time the gain sweep (100 signals, after a warm-up call that compiles the
Numba kernels).

In [ ]:
def bloch_gain(an, **kwargs):
    return an.gain(SIG, pump=FP, model="bloch", modes=MODES, bloch_config=CFG, **kwargs)


an_uni = analysis(base)
bloch_gain(an_uni)  # warm-up (compilation)
t0 = time.perf_counter()
r_uni = bloch_gain(an_uni)
t_uni = time.perf_counter() - t0
print(
    f"uniform: max gain {r_uni.gain_db.max():.2f} dB, {t_uni:.2f} s, "
    f"not converged at {SIG[~r_uni.converged]} GHz"
)

n_list = [1, 5, 20, 60, 200]
timing, dev = [], []
for n in n_list:
    a = analysis(make_segmented(n))
    a.linear  # linear response outside the timing
    times = []
    for _ in range(2):  # best of two runs
        t0 = time.perf_counter()
        r = bloch_gain(a)
        times.append(time.perf_counter() - t0)
    timing.append(min(times))
    dev.append(np.max(np.abs(r.gain_db - r_uni.gain_db)))
    print(
        f"n_seg={n:4d} ({base.N // n:4d} supercells each): {timing[-1]:5.2f} s, "
        f"max |gain - uniform| = {dev[-1]:.3f} dB, {np.sum(~r.converged)} not converged"
    )

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.semilogx(n_list, timing, "o-")
ax.set_ylim(bottom=0)
ax.set(
    xlabel="number of segments",
    ylabel="time per gain sweep (s)",
    title=f"{len(SIG)} signals, bloch model",
)
fig.tight_layout()

The identical-segment results agree with the uniform device to about $10^{-3}$ dB. What remains comes
from the tolerance of the forward-backward iteration (`convergence_threshold`, a relative residual of
$10^{-2}$: each run stops at a slightly different point) and from interpolating the stored envelopes
between the output points (set by `thin`). Points that do not converge within the 60 passes would be
printed above; treat those with care.

The cost grows slowly with the number of segments: the integrator is restarted and the mixing terms
are rebuilt in every segment, but segments with identical unit cells share the same Bloch tables,
which are computed only once. Wall-clock timings on a shared machine are noisy; take the plot as an
order of magnitude.

## 3. Fabrication variations

Now every segment gets its own random $L$ and $L_f$: each is multiplied by $1+\sigma\,\mathcal N(0,1)$,
independently per segment and per parameter, with $\sigma = 2\%$ and 40 segments of 30 supercells.
$L$ sets the impedance and the phase velocity of the line, so its steps reflect; $L_f$ is in the
fingers' shunt branch and mostly changes the dispersion at high frequency, around the stopbands.
Three seeds give three "chips".

In [ ]:
n_seg, sigma = 40, 0.02
chips = {}
for seed in [1, 2, 3]:
    rng = np.random.default_rng(seed)
    L_scale = 1 + sigma * rng.standard_normal(n_seg)
    Lf_scale = 1 + sigma * rng.standard_normal(n_seg)
    a = analysis(make_segmented(n_seg, L_scale, Lf_scale))
    r = bloch_gain(a)
    chips[seed] = {"L": L_scale, "Lf": Lf_scale, "an": a, "r": r}
    f_ps = FP + SIG[-1]  # p+s at the highest signal
    in_gap = np.sum(a.linear.alpha[:, np.argmin(np.abs(a.linear.freqs - f_ps))] > 1e-4)
    print(
        f"seed {seed}: max gain {np.nanmax(r.gain_db):.2f} dB (nominal {r_uni.gain_db.max():.2f} dB), "
        f"{np.sum(~r.converged)} points not converged; p+s = {f_ps:.1f} GHz in the stopband of "
        f"{in_gap}/{n_seg} segments, S21 there {r.gain_db[-1]:.1f} dB"
    )

In [ ]:
f = an_uni.linear.freqs
win = (f > 12.8) & (f < 15.0)
colors = ["C0", "C1", "C2"]
fig, axs = plt.subplots(2, 2, figsize=(13, 8))
edges = np.linspace(0, 1, n_seg + 1)
for c, (seed, d) in zip(colors, chips.items()):
    axs[0, 0].stairs(
        100 * (d["L"] - 1), edges, baseline=None, color=c, label=f"seed {seed}"
    )
    axs[0, 0].stairs(100 * (d["Lf"] - 1), edges, baseline=None, color=c, ls=":")
axs[0, 0].set(
    xlabel="position $x/\\ell$",
    ylabel="change (%)",
    title="$L$ (solid) and $L_f$ (dotted)",
)
axs[0, 0].legend(fontsize=9)

# Attenuation per unit cell of every segment of chip 1: nonzero inside each segment's stopband.
lin1 = chips[1]["an"].linear
for j, alpha in enumerate(lin1.alpha):
    axs[0, 1].plot(f[win], alpha[win], color=plt.cm.viridis(j / n_seg), lw=0.8)
axs[0, 1].plot(f[win], an_uni.linear.alpha[0][win], "k--", lw=1.5, label="nominal")
axs[0, 1].axvline(FP, color="r", ls=":", label="pump")
axs[0, 1].set(
    xlabel="frequency (GHz)",
    ylabel="alpha (1 / unit cell)",
    title="first stopband of each segment, seed 1",
)
axs[0, 1].legend(fontsize=9)

axs[1, 0].plot(SIG, r_uni.gain_db, "k", lw=2, label="nominal")
for c, (seed, d) in zip(colors, chips.items()):
    r = d["r"]
    axs[1, 0].plot(SIG, r.gain_db, color=c, label=f"seed {seed}")
    axs[1, 0].plot(SIG[~r.converged], r.gain_db[~r.converged], "x", color=c)
axs[1, 0].set(
    xlabel="signal frequency (GHz)",
    ylabel="$S_{21}$ (dB)",
    title="gain (x: not converged)",
)
axs[1, 0].legend(fontsize=9)

axs[1, 1].plot(SIG, r_uni.gain_db_11, "k", lw=2, label="nominal")
for c, (seed, d) in zip(colors, chips.items()):
    axs[1, 1].plot(SIG, d["r"].gain_db_11, color=c, label=f"seed {seed}")
axs[1, 1].set(
    xlabel="signal frequency (GHz)",
    ylabel="$S_{11}$ (dB)",
    title="signal reflection at the input",
)
axs[1, 1].legend(fontsize=9)
fig.tight_layout()

A few-percent spread moves the stopband of each segment by a few hundred MHz (top right): the
stopband frequency scales roughly as $1/\sqrt{LC}$, and the pump at 15.8 GHz sits 1.5 GHz above its upper
edge, where the dispersion is strong. The pump wavenumber, and with it the phase matching, then changes
from segment to segment. For this spread the gain in the band changes by up to about 0.7 dB from chip to
chip (bottom left). The input
reflection $S_{11}$ under pump (bottom right) is dominated by the amplified signal reflected at the
device ends, which the nominal device has too; the steps between segments change it from chip to
chip.

Modes far from the signal band are sensitive too: at the highest signal frequency the sum-frequency
mode $p+s$ lies just above the second stopband of the nominal device, and the spread puts it inside the
stopband of some segments (counted in the output above). The signal transmission at that point then differs strongly
from chip to chip.

Modes inside the stopbands of many segments are also the hardest case for the forward-backward
iteration: the reflections between segments couple their two directions strongly. They are solved
on the fixed edge basis, and here the iteration converges for every chip and signal; a point that
does not converge is flagged in `GainResult.converged` (a cross in the gain plot) and should not be
trusted. Note that the plotted stopbands come from `an.linear.alpha` per segment, while
`an.linear.stopbands` only reports those of the first segment.

## 4. A Floquet-mode TWPA from segments

In a uniformly pumped TWPA the signal does not only turn into idler photons: the pump also converts it to sidebands
(here $f_p \pm f_s$, $2 f_p \pm f_s$ and the corresponding idler terms). Photons left in those modes at the output
are lost information, which lowers the quantum efficiency. Peng *et al.* showed that switching the nonlinearity on
and off adiabatically along the line maps the input and output linear modes onto single Floquet modes of the pumped
line, so that the sideband content at the output nearly vanishes. The taper does not phase match anything; it cleans
the output.

We cut the holotype into 40 segments and scale its nonlinearity
($\propto 1/I_*^2$) with a Gaussian profile that keeps 10 % at the ends. The linear circuit is the same in every
segment, so the taper adds no reflections.

In [ ]:
n_seg, floor, sigma = 40, 0.1, 0.2
u = (np.arange(n_seg) + 0.5) / n_seg
profile = floor + (1 - floor) * np.exp(-((u - 0.5) ** 2) / (2 * sigma**2))


def floquet_line(Ip0: float) -> TWPA:
    segments = [
        base.model_copy(
            deep=True,
            update={"N": base.N // n_seg, "Istar": base.Istar / np.sqrt(p), "Ip0": Ip0},
        )
        for p in profile
    ]
    return TWPA(cells=segments, Z0_ref=base.Z0_ref, Ip0=Ip0)


plt.figure(figsize=(5, 2.5))
plt.step(u, profile, where="mid")
plt.xlabel("Position along the line")
plt.ylabel("Relative nonlinearity")
plt.show()

The weaker average nonlinearity is compensated by a stronger pump (0.181 mA instead of 0.1 mA), which gives the
same mean gain. We keep the pump harmonics and two orders of frequency conversion, solve with the `bloch` model and
count, at the output, the photon flux $\propto |I|^2/f$ in the conversion modes relative to signal plus idler.

In [ ]:
LOST = ("ps", "pi", "p2s", "p2i")
signals = np.linspace(4, 11, 75)
results = {}
for name, twpa in (
    ("uniform", base.model_copy(update={"Ip0": 1e-4})),
    ("Floquet", floquet_line(1.809e-4)),
):
    an = analysis(twpa)
    r = an.gain(
        signals, pump=FP, Is0=1e-9, model="bloch", modes=MODES, bloch_config=CFG
    )
    f = an.problem(MODES, {"p": FP, "s": signals}).freqs
    flux = np.abs(r.b_out) ** 2 / np.abs(f)
    lost = sum(flux[:, r.mode(m)] for m in LOST)
    useful = flux[:, r.mode("s")] + flux[:, r.mode("i")]
    results[name] = (r, lost / useful)
    print(
        f"{name}: mean gain {np.mean(r.gain_db):.1f} dB, lost / (signal + idler) photons "
        f"{10 * np.log10(np.mean(lost / useful)):.1f} dB, converged {r.converged.sum()}/{len(signals)}"
    )

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.5))
for name, (r, ratio) in results.items():
    ax1.plot(r.signal_freqs, r.gain_db, label=name)
    ax2.plot(r.signal_freqs, 10 * np.log10(ratio), label=name)
ax1.set_ylabel("Gain (dB)")
ax2.set_ylabel("Lost / useful photons (dB)")
for ax in (ax1, ax2):
    ax.set_xlabel("Signal frequency (GHz)")
    ax.legend()
plt.tight_layout()
plt.show()

At equal mean gain the tapered line leaves about 16 times (12 dB) fewer photons in the conversion modes. The mode
currents along the line show why: in the uniform device the conversion modes keep growing up to the output, while
in the Floquet device they are populated in the middle and return to the signal and idler as the nonlinearity
switches off.

In [ ]:
for name, (r, _) in results.items():
    ax = r.plot_currents(title=f"{name}: mode currents, averaged over the signals")
plt.show()